In [1]:
!pip install TTS

  Using cached tts-0.22.0-cp310-cp310-macosx_15_0_arm64.whl
  Using cached numpy-1.22.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.0 kB)
  Using cached inflect-7.5.0-py3-none-any.whl.metadata (24 kB)
  Using cached anyascii-0.3.3-py3-none-any.whl.metadata (1.6 kB)
  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached pandas-1.5.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached trainer-0.0.36-py3-none-any.whl.metadata (8.1 kB)
  Using cached coqpit-0.0.17-py3-none-any.whl.metadata (11 kB)
  Using cached jieba-0.42.1-py3-none-any.whl
  Using cached pypinyin-0.55.0-py2.py3-none-any.whl.metadata (12 kB)
  Using cached hangul_romanize-0.1.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached gruut-2.2.3-py3-none-any.whl
  Using cached jamo-0.4.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached g2pkk-0.1.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached bangla-0.0.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached bnnumerizer-0.0.2-py3-none-any.w

In [1]:
import torch

# Monkey-patch torch.load to always use weights_only=False
# Use this ONLY if you trust the source (Coqui XTTS is safe)
original_load = torch.load
torch.load = lambda *args, **kwargs: original_load(*args, **{**kwargs, 'weights_only': False})

from TTS.api import TTS
xtts_model = TTS('tts_models/multilingual/multi-dataset/xtts_v2')


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.


/Users/robbannn/.pyenv/versions/3.10.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 > Using model: xtts


GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [ ]:
from TTS.api import TTS

# This will return the local paths of the model files (it won't redownload if they already exist)
model_path, config_path, vocoder_path, vocoder_config_path, model_dir = TTS().download_model_by_name("tts_models/multilingual/multi-dataset/xtts_v2")

print(f"Model Directory: {model_dir}")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
Model Directory: /Users/robbannn/Library/Application Support/tts/tts_models--multilingual--multi-dataset--xtts_v2


In [5]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config = XttsConfig()
# Use the absolute path you already used for the config
model_path = "/Users/robbannn/Desktop/PROJECTS/urduTTS/models/xtts_v2/"

config.load_json(model_path + "config.json")

model = Xtts.init_from_config(config)

# Pass the directory path, NOT the model.pth file
model.load_checkpoint(config, checkpoint_dir=model_path, eval=True)


In [ ]:
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts

#  PATHS 
XTTS_CHECKPOINT = "/Users/robbannn/Desktop/PROJECTS/urduTTS/models/xtts_v2/"
DATASET_PATH    = "/Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset//"
OUTPUT_PATH     = "/Users/robbannn/Desktop/PROJECTS/urduTTS/xtts_dataset/output/"

# CONFIG 
config = XttsConfig()
config.load_json(XTTS_CHECKPOINT + "config.json")

config.output_path = OUTPUT_PATH
config.epochs      = 20
config.batch_size  = 4
config.eval_batch_size = 2
config.lr          = 5e-6

In [ ]:
# DATASET 
train_samples, eval_samples = load_tts_samples(
    datasets=[{
        "formatter"  : "ljspeech",
        "dataset_name": "urdu_tts",
        "path"       : DATASET_PATH,
        "meta_file_train": "train/metadata.csv",
        "meta_file_val"  : "val/metadata.csv",
        "language"   : "ur",
        "ignored_speakers": None,
    }],
    eval_split=True,
)

In [ ]:
# MODEL 
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=XTTS_CHECKPOINT, eval=False)

In [ ]:
# FREEZE 
for name, param in model.named_parameters():
    param.requires_grad = False                   # freeze everything first

for name, param in model.named_parameters():
    if "gpt" in name:                             # unfreeze GPT-2 top layers
        param.requires_grad = True
    if "text_embedding" in name:                  # unfreeze text embeddings
        param.requires_grad = True

# TRAIN 
trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()

In [ ]:
import torch
import torch.nn as nn
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts


In [ ]:
# CUSTOM ADAPTER 
class UrduAdapter(nn.Module):
    """
    Bottleneck Adapter for Urdu language adaptation.
    Sits between Text Embedding and GPT-2.
    """
    def __init__(self, input_dim=512, bottleneck_dim=128):
        super().__init__()

        self.adapter = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim),   # compress
            nn.LayerNorm(bottleneck_dim),            # normalize
            nn.ReLU(),                               # activate
            nn.Linear(bottleneck_dim, input_dim),   # expand back
        )

        # residual scale — learnable, starts small
        self.scale = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        return x + self.scale * self.adapter(x)    # residual connection

In [ ]:

# XTTS +  ADAPTER COMBINED 
class XttsWithUrduAdapter(nn.Module):
    """
    Wraps XTTS and injects the UrduAdapter into the forward pass.
    """
    def __init__(self, xtts_model, adapter):
        super().__init__()
        self.xtts    = xtts_model
        self.adapter = adapter

    def forward(self, text_inputs, *args, **kwargs):
        # Step 1 → get text embeddings from XTTS
        embeddings = self.xtts.text_encoder(text_inputs)

        # Step 2 → pass through YOUR adapter
        adapted_embeddings = self.adapter(embeddings)

        # Step 3 → rest of XTTS runs normally
        return self.xtts.gpt(adapted_embeddings, *args, **kwargs)

In [ ]:


# PATHS 
XTTS_CHECKPOINT = "path/to/xtts_v2/"
DATASET_PATH    = "xtts_dataset/"
OUTPUT_PATH     = "xtts_dataset/output/"


# LOAD BASE XTTS 
config = XttsConfig()
config.load_json(XTTS_CHECKPOINT + "config.json")

config.output_path     = OUTPUT_PATH
config.epochs          = 20
config.batch_size      = 4
config.eval_batch_size = 2
config.lr              = 5e-6

base_model = Xtts.init_from_config(config)
base_model.load_checkpoint(
    config,
    checkpoint_dir=XTTS_CHECKPOINT,
    eval=False
)

In [ ]:

# FREEZE / UNFREEZE STRATEGY 
# EXP1 => freeze everything in base XTTS
for param in base_model.parameters():
    param.requires_grad = False

# EXP => selectively unfreeze for experimentation
for name, param in base_model.named_parameters():
    if "gpt" in name:            # unfreeze GPT-2 top layers
        param.requires_grad = True
    if "text_embedding" in name: # unfreeze text embeddings
        param.requires_grad = True

In [ ]:
# ADAPTER 
# Adapter is always trainable 
adapter = UrduAdapter(input_dim=512, bottleneck_dim=128)

In [ ]:
# combining both
model = XttsWithUrduAdapter(xtts_model=base_model, adapter=adapter)

In [ ]:
# DATASET 
train_samples, eval_samples = load_tts_samples(
    datasets=[{
        "formatter"       : "ljspeech",
        "dataset_name"    : "urdu_tts",
        "path"            : DATASET_PATH,
        "meta_file_train" : "train/metadata.csv",
        "meta_file_val"   : "val/metadata.csv",
        "language"        : "ur",
        "ignored_speakers": None,
    }],
    eval_split=True,
)

In [ ]:
# TRAIN 
trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

In [ ]:
trainer.fit()